In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
from pathlib import Path

import pandas as pd
from jppype import Mosaic, vscode_theme
from tqdm import tqdm

from fundus_toolkits import FundusData
from fundus_toolkits.utils.data_io import most_common_image_ext
from fundus_vessels_toolkit.models.topology.dataset import BranchDigraphDataset, SampleInfo

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [12]:
DATASETS_ROOT = Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/")
DATASETS_PATH = {
    dataset: DATASETS_ROOT / folder
    for dataset, folder in {
        "GAVE-train": "GAVE-train",
        "MAPLES-DR": "MAPLES-DR",
        "FundusAV": "Fundus-AV",
        "HRF": "HRF",
        "LES-AV": "LES-AV",
        "INSPIRE": "INSPIRE",
        "DRIVE_train": "AV_DRIVE/training",
        "DRIVE_test": "AV_DRIVE/test",
    }.items()
}

## Preprocess Datasets


In [14]:
from wrapper_automorph import automorph_segment_av
from wrapper_vascx import vascx_segment_av

from fundus_vessels_toolkit.models import segment_av


for dataset_path in DATASETS_PATH.values():
    raw_path = dataset_path / "1-images"
    for raw_file in tqdm(Path(raw_path).glob(f"*{most_common_image_ext(raw_path)}")):
        fvt_out = dataset_path / "2-av-pred_FVT" / (raw_file.stem + ".png")
        automorph_out = dataset_path / "2-av-pred_Automorph" / (raw_file.stem + ".png")
        vascx_out = dataset_path / "2-av-pred_VascX" / (raw_file.stem + ".png")

        if fvt_out.exists() and automorph_out.exists() and vascx_out.exists():
            continue

        fundus = FundusData(image=raw_file)
        fundus_cropped, roi = fundus.crop_to_roi(return_roi=True)

        if not fvt_out.exists():
            segment_av(fundus_cropped)
            fundus.update(av=fundus_cropped.av, roi=roi).write_image(av=fvt_out)

        if not automorph_out.exists():
            automorph_segment_av(fundus_cropped)
            fundus.update(av=fundus_cropped.av, roi=roi).write_image(av=automorph_out)

        if not vascx_out.exists():
            vascx_segment_av(fundus_cropped)
            fundus.update(av=fundus_cropped.av, roi=roi).write_image(av=vascx_out)

0it [00:00, ?it/s]/home/gaby/.conda/envs/lab/lib/python3.13/site-packages/monai/inferers/utils.py:226: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:309.)
  win_data = torch.cat([inputs[win_slice] for win_slice in unravel_slice]).to(sw_device)
/home/gaby/.conda/envs/lab/lib/python3.13/site-packages/monai/inferers/utils.py:370: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc

In [11]:
RAW = [path / "1-images" for path in DATASETS_PATH.values()]
TOPO = [path / "3-topo" for path in DATASETS_PATH.values()]
AV = [
    {
        "fvt": path / "2-av-pred_FVT",
        "automorph": path / "2-av-pred_Automorph",
        "gt": path / "2-av",
        "vascx": path / "2-av-pred_VascX",
    }
    for path in DATASETS_PATH.values()
]

dataset = BranchDigraphDataset.load_from_dirs(
    RAW,
    TOPO,
    AV,
    dataset_name=list(DATASETS_PATH.keys()),
    resize_to=1024,
    output_dir="tmp/ALL_DATA",
    n_workers=0,
    mask_optic_disc=True,
)

NameError: name 'DATASETS_PATH' is not defined

## Define dataset splits


In [9]:
samples_by_dataset: dict[str, list[SampleInfo]] = {}
for sample in dataset.samples_info:
    samples_by_dataset.setdefault(sample.dataset, []).append(sample)

samples_stratification: dict[str, list[SampleInfo]] = {}


**Fundus AV**: stratify by pathology


In [10]:
for sample in samples_by_dataset["FundusAV"]:
    samples_stratification.setdefault(f"FundusAV-{sample.name[-1]}", []).append(sample)

**LES-AV**, **INSPIRE** and **GAVE-train**: no specific stratification


In [11]:
samples_stratification["LES-AV"] = samples_by_dataset["LES-AV"]
samples_stratification["INSPIRE"] = samples_by_dataset["INSPIRE"]
samples_stratification["GAVE-train"] = samples_by_dataset["GAVE-train"]
samples_stratification["HRF"] = samples_by_dataset["HRF"]

In [12]:
for samples in samples_stratification.values():
    N = len(samples)
    train_end, val_end = int(N * 0.7), int(N * 0.85)
    for sample in samples[:train_end]:
        sample.dataset_type = "train"
    for sample in samples[train_end:val_end]:
        sample.dataset_type = "validation"
    for sample in samples[val_end:]:
        sample.dataset_type = "test"

Use existing splits for **DRIVE** and **MAPLES-DR**


In [13]:
for sample in samples_by_dataset["DRIVE_train"]:
    sample.dataset_type = "validation" if sample.name.startswith(("31", "33", "35", "28")) else "train"
for sample in samples_by_dataset["DRIVE_test"]:
    sample.dataset_type = "test"

In [14]:
import maples_dr

maples_dr_test_samples_name = {s.name for s in maples_dr.load_test_set()}
maples_dr_test_samples: list[SampleInfo] = []
for sample in samples_by_dataset["MAPLES-DR"]:
    if sample.name in maples_dr_test_samples_name:
        maples_dr_test_samples.append(sample)
    else:
        sample.dataset_type = "train"
N_test = len(maples_dr_test_samples)
for sample in maples_dr_test_samples[: N_test // 2]:
    sample.dataset_type = "validation"
for sample in maples_dr_test_samples[N_test // 2 :]:
    sample.dataset_type = "test"

Thanks for using MAPLES-DR!
  When using this dataset in academic works,
  please cite: ]8;id=256913;https://doi.org/10.1038/s41597-024-03739-6\https://doi.org/10.1038/s41597-024-03739-6]8;;\

Check that all samples have a dataset type assigned and save the splits


In [15]:
samples_without_type = [
    sample for sample in dataset.samples_info if sample.dataset_type not in ("train", "validation", "test")
]
assert not samples_without_type, f"Samples without dataset type: {[s.name for s in samples_without_type]}"

dataset.save_manifest()

In [16]:
train_set, val_set, test_set = dataset.split_sets()


def count_dataset(dataset):
    counts = {}
    for sample in dataset.samples_info:
        counts[sample.dataset] = counts.get(sample.dataset, 0) + 1
    return counts


df = pd.DataFrame(
    {"train": count_dataset(train_set), "validation": count_dataset(val_set), "test": count_dataset(test_set)}
).T
df["TOTAL"] = df.sum(axis=1)
df

,FundusAV,HRF,LES-AV,MAPLES-DR,DRIVE_train,GAVE-train,INSPIRE,DRIVE_test,TOTAL
train,70.0,28.0,15.0,73.0,16.0,35.0,10.0,NaN,247.0
validation,15.0,6.0,3.0,16.0,4.0,7.0,2.0,NaN,53.0
test,15.0,7.0,4.0,16.0,NaN,8.0,3.0,20.0,73.0


## Bundle dataset


In [2]:
dataset.bundle("ALL_DATA_bundle.tar.gz", overwrite=True)

NameError: name 'dataset' is not defined

In [3]:
bundled_dataset = BranchDigraphDataset("ALL_DATA_bundle.tar.gz")

Processing...
Done!


In [4]:
train_set, val_set, test_set = bundled_dataset.split_sets()


def count_dataset(dataset):
    counts = {}
    for sample in dataset.samples_info:
        counts[sample.dataset] = counts.get(sample.dataset, 0) + 1
    return counts


df = pd.DataFrame(
    {"train": count_dataset(train_set), "validation": count_dataset(val_set), "test": count_dataset(test_set)}
).T
df["TOTAL"] = df.sum(axis=1)
df

,FundusAV,HRF,LES-AV,MAPLES-DR,DRIVE_train,GAVE-train,INSPIRE,DRIVE_test,TOTAL
train,70.0,28.0,15.0,73.0,16.0,35.0,10.0,NaN,247.0
validation,15.0,6.0,3.0,16.0,4.0,7.0,2.0,NaN,53.0
test,15.0,7.0,4.0,16.0,NaN,8.0,3.0,20.0,73.0


## Test


In [10]:
m, sample, sample_data = val_set.jppype_show(51, augment=False, version="fvt")

m

/home/gaby/Lab/Libs/jppype/jppype/layers/layers_2d.py:594: RuntimeWarning: invalid value encountered in cast
  self._nodes_coordinates = node_yx.astype(np.uint32)


GridBox(children=(HTML(value='<h3 style="text-align: center;">image33/fvt</h3>'), HTML(value='<h3 style="text-…